In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import plotnine as gg
from scvi.model import SCVI

from essential.utils import PLOTNINE_DEFAULT_THEME_2

In [ ]:
adata = sc.read_h5ad("/workspace/data/de122_lce75/adata_de122_lce75_merged.h5ad")


In [ ]:
np.median(adata.layers["reads"].sum(1).A1)

In [ ]:
np.median(adata.layers["reads"].sum(0).A1)

In [ ]:
n_counts = np.asarray(adata.layers["reads"].sum(1)).flatten()
adata.obs["n_counts_reads"] = n_counts
adata.obs.groupby("target")["n_counts_reads"].sum().median()

In [ ]:
adata.obs.groupby("target")["n_counts_reads"].median().sort_values()

In [ ]:
adata.obs[["spacer", "batch"]].value_counts()

In [ ]:
SCVI.setup_anndata(
    adata, batch_key="rt_bc", layer="reads", categorical_covariate_keys=["batch"]
)
model = SCVI(adata)
model.train()

In [ ]:
latent = model.get_latent_representation()

In [ ]:
adata.obsm["X_scvi"] = latent
sc.pp.neighbors(adata, use_rep="X_scvi")
sc.tl.umap(adata)
sc.pl.umap(adata)

adata.obs["UMAP1"] = adata.obsm["X_umap"][:, 0]
adata.obs["UMAP2"] = adata.obsm["X_umap"][:, 1]

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scvi")
sc.tl.leiden(adata, key_added="leiden_1.0", resolution=1.0)
sc.tl.leiden(adata, key_added="leiden_0.5", resolution=0.5)
sc.tl.leiden(adata, key_added="leiden_0.1", resolution=0.1)

In [ ]:
import seaborn as sns

n = len(adata.obs["leiden_0.5"].unique())
palette = sns.color_palette("hls", n_colors=n).as_hex()

fig = (
    gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2", fill="leiden_0.5"))
    + gg.geom_point(color="black")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.scale_fill_manual(values=palette)
)
fig

In [ ]:
# compute cluster centroids for labels
centroids = adata.obs.groupby("leiden_0.5")[["UMAP1", "UMAP2"]].mean().reset_index()

fig = (
    gg.ggplot(adata.obs, gg.aes(x="UMAP1", y="UMAP2"))
    + gg.geom_point(color="grey", fill="grey", stroke=0.0, size=1.0, alpha=0.5)
    + gg.geom_text(centroids, gg.aes(label="leiden_0.5"), size=8, color="black")
    + PLOTNINE_DEFAULT_THEME_2
)
fig

In [ ]:
sc.tl.tsne(adata, use_rep="X_scvi", perplexity=50)
adata.obs["TSNE1"] = adata.obsm["X_tsne"][:, 0]
adata.obs["TSNE2"] = adata.obsm["X_tsne"][:, 1]

In [ ]:
is_nontargeting = adata.obs["target"].str.startswith("nontargeting")
clusters_with_controls = set(adata.obs.loc[is_nontargeting, "leiden_0.1"].unique())
all_clusters = set(adata.obs["leiden_0.1"].unique())
case_only_clusters = sorted(all_clusters - clusters_with_controls, key=int)
print(case_only_clusters)

In [ ]:
case_only_clusters = ["7", "10", "6", "8", "9", "10", "11", "12"]
adata.obs["control-like"] = adata.obs["leiden_0.5"].isin(case_only_clusters)

# case_only_clusters = ["2", "3", "4"]
# adata.obs["control-like"] = adata.obs["leiden_0.1"].isin(case_only_clusters)
adata.obs["control-like"] = adata.obs["control-like"].map(
    {True: "other", False: "control-like"}
)

In [ ]:
rep_x = "TSNE1"
rep_y = "TSNE2"

x_min, x_max = adata.obs[rep_x].min(), adata.obs[rep_x].max()
y_min, y_max = adata.obs[rep_y].min(), adata.obs[rep_y].max()
x_range, y_range = x_max - x_min, y_max - y_min

ox = x_min - x_range * 0.02
oy = y_min - y_range * 0.02
arrow_x, arrow_y = x_range * 0.15, y_range * 0.15

arrow_df = pd.DataFrame(
    {
        "x": [ox, ox],
        "y": [oy, oy],
        "xend": [ox + arrow_x, ox],
        "yend": [oy, oy + arrow_y],
    }
)
label_df = pd.DataFrame(
    {
        "x": [ox + arrow_x / 2, ox - x_range * 0.05],
        "y": [oy - y_range * 0.05, oy + arrow_y / 2],
        "label": [rep_x, rep_y],
        "angle": [0, 90],
    }
)

fig = (
    gg.ggplot(adata.obs, gg.aes(x=rep_x, y=rep_y))
    + gg.geom_point(gg.aes(fill="control-like"), color="black", stroke=0.005, size=0.5)
    + gg.geom_point(
        adata.obs[adata.obs["target"].str.startswith("nontargeting")],
        color="black",
        size=1.0,
        stroke=0.0,
    )
    + gg.scale_fill_manual(values={"control-like": "#C8C8C8", "other": "#BF6E6E"})
    + gg.geom_segment(
        arrow_df,
        gg.aes(x="x", y="y", xend="xend", yend="yend"),
        arrow=gg.arrow(length=0.08, type="closed"),
        color="black",
        size=0.4,
    )
    + gg.geom_text(
        label_df,
        gg.aes(x="x", y="y", label="label", angle="angle"),
        size=8,
        color="black",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        axis_title=gg.element_blank(),
        axis_text=gg.element_blank(),
        axis_ticks=gg.element_blank(),
        axis_line=gg.element_blank(),
    )
    + gg.guides(fill=gg.guide_legend(override_aes={"size": 2}))
)
fig.save(
    "/workspace/experiments/06152026_retreat_content/umap_control_like.png", dpi=700
)
fig

In [ ]:
adata_case = adata[adata.obs["control-like"] == "other"].copy()
SCVI.setup_anndata(
    adata_case, batch_key="rt_bc", layer="reads", categorical_covariate_keys=["batch"]
)
model = SCVI(adata_case)
model.train()

In [ ]:
latent = model.get_latent_representation()
adata_case.obsm["X_scvi"] = latent
sc.pp.neighbors(adata_case, use_rep="X_scvi", n_neighbors=5)
sc.tl.umap(adata_case, min_dist=0.5)
sc.pl.umap(adata_case)

adata_case.obs["CASE_UMAP1"] = adata_case.obsm["X_umap"][:, 0]
adata_case.obs["CASE_UMAP2"] = adata_case.obsm["X_umap"][:, 1]

In [ ]:
sc.tl.tsne(adata_case, use_rep="X_scvi", perplexity=30)
adata_case.obs["CASE_TSNE1"] = adata_case.obsm["X_tsne"][:, 0]
adata_case.obs["CASE_TSNE2"] = adata_case.obsm["X_tsne"][:, 1]

In [ ]:
sc.pp.neighbors(adata_case, use_rep="X_scvi", n_neighbors=15)
sc.tl.leiden(adata_case, key_added="leiden", resolution=1.0)

In [ ]:
adata_case.obs.keys()

In [ ]:
xrep = "CASE_TSNE1"
yrep = "CASE_TSNE2"

x_min, x_max = adata_case.obs[xrep].min(), adata_case.obs[xrep].max()
y_min, y_max = adata_case.obs[yrep].min(), adata_case.obs[yrep].max()
x_range, y_range = x_max - x_min, y_max - y_min

ox = x_min - x_range * 0.02
oy = y_min - y_range * 0.02
arrow_x, arrow_y = x_range * 0.15, y_range * 0.15

arrow_df = pd.DataFrame(
    {
        "x": [ox, ox],
        "y": [oy, oy],
        "xend": [ox + arrow_x, ox],
        "yend": [oy, oy + arrow_y],
    }
)
label_df = pd.DataFrame(
    {
        "x": [ox + arrow_x / 2, ox - x_range * 0.05],
        "y": [oy - y_range * 0.05, oy + arrow_y / 2],
        "label": [xrep, yrep],
        "angle": [0, 90],
    }
)

fig = (
    gg.ggplot(adata_case.obs, gg.aes(x=xrep, y=yrep))
    + gg.geom_point(gg.aes(fill="leiden"), color="black", stroke=0.005, size=0.5)
    + gg.geom_segment(
        arrow_df,
        gg.aes(x="x", y="y", xend="xend", yend="yend"),
        arrow=gg.arrow(length=0.08, type="closed"),
        color="black",
        size=0.4,
    )
    + gg.geom_text(
        label_df,
        gg.aes(x="x", y="y", label="label", angle="angle"),
        size=8,
        color="black",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        axis_title=gg.element_blank(),
        axis_text=gg.element_blank(),
        axis_ticks=gg.element_blank(),
        axis_line=gg.element_blank(),
    )
    + gg.labs(
        x="TSNE1",
        y="TSNE2",
    )
    + gg.guides(fill=gg.guide_legend(override_aes={"size": 2}))
)
fig

In [ ]:
for cluster in adata_case.obs["leiden"].unique():
    print(cluster)
    represented_targets = (
        adata_case.obs.loc[adata_case.obs["leiden"] == cluster, "target"]
        .value_counts()
        .loc[lambda x: x >= 2]
    )
    # print(", ".join(represented_targets.index))
    print(represented_targets)

In [ ]:
mapper = {
    "0": "Other metabolism",
    "1": "Other metabolism",
    "2": "Cell envelope",
    "3": "Central metabolism",
    "4": "Cell envelope",
    "5": "Ribosome & translation",
    "6": "Respiration",
    "7": "Other",
    "8": "Ribosome & translation",
    "9": "DNA replication",
    "10": "Other",
    "11": "Ribosome & translation",
    "12": "Other",
    "13": "Cell envelope",
    "14": "Central metabolism",
    "15": "Metal & nutrient homeostasis",
    "16": "Ribosome & translation",
    "17": "Metal & nutrient homeostasis",
    "18": "Respiration",
    "19": "Cell envelope",
    "20": "Transcription & RNA processing",
    "21": "Cell envelope",
    "22": "Transcription & RNA processing",
    "23": "Respiration",
    "24": "Protein export & folding",
    "25": "Protein export & folding",
    "26": "Cell envelope",
    "27": "Ribosome & translation",
    "28": "Prophage",
    "29": "Empty",
}

In [ ]:
adata_case.obs["annotated_cluster"] = adata_case.obs["leiden"].map(mapper)

In [ ]:
xrep = "CASE_TSNE1"
yrep = "CASE_TSNE2"

x_min, x_max = adata_case.obs[xrep].min(), adata_case.obs[xrep].max()
y_min, y_max = adata_case.obs[yrep].min(), adata_case.obs[yrep].max()
x_range, y_range = x_max - x_min, y_max - y_min

ox = x_min - x_range * 0.02
oy = y_min - y_range * 0.02
arrow_x, arrow_y = x_range * 0.15, y_range * 0.15

arrow_df = pd.DataFrame(
    {
        "x": [ox, ox],
        "y": [oy, oy],
        "xend": [ox + arrow_x, ox],
        "yend": [oy, oy + arrow_y],
    }
)
label_df = pd.DataFrame(
    {
        "x": [ox + arrow_x / 2, ox - x_range * 0.05],
        "y": [oy - y_range * 0.05, oy + arrow_y / 2],
        "label": [xrep, yrep],
        "angle": [0, 90],
    }
)

fig = (
    gg.ggplot(adata_case.obs, gg.aes(x=xrep, y=yrep))
    + gg.geom_point(
        gg.aes(fill="annotated_cluster"), color="black", stroke=0.005, size=0.5
    )
    + gg.geom_segment(
        arrow_df,
        gg.aes(x="x", y="y", xend="xend", yend="yend"),
        arrow=gg.arrow(length=0.08, type="closed"),
        color="black",
        size=0.4,``
    )
    + gg.geom_text(
        label_df,
        gg.aes(x="x", y="y", label="label", angle="angle"),
        size=8,
        color="black",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        axis_title=gg.element_blank(),
        axis_text=gg.element_blank(),
        axis_ticks=gg.element_blank(),
        axis_line=gg.element_blank(),
    )
    + gg.labs(
        x="TSNE1",
        y="TSNE2",
    )
    + gg.guides(fill=gg.guide_legend(override_aes={"size": 2}))
)
fig

In [ ]:
adata_case.write_h5ad(
    "/workspace/experiments/06152026_retreat_content/adata_case_deeper_sequencing.h5ad"
)

In [ ]:
xrep = "CASE_TSNE1"
yrep = "CASE_TSNE2"

x_min, x_max = adata_case.obs[xrep].min(), adata_case.obs[xrep].max()
y_min, y_max = adata_case.obs[yrep].min(), adata_case.obs[yrep].max()
x_range, y_range = x_max - x_min, y_max - y_min

ox = x_min - x_range * 0.02
oy = y_min - y_range * 0.02
arrow_x, arrow_y = x_range * 0.15, y_range * 0.15

arrow_df = pd.DataFrame(
    {
        "x": [ox, ox],
        "y": [oy, oy],
        "xend": [ox + arrow_x, ox],
        "yend": [oy, oy + arrow_y],
    }
)
label_df = pd.DataFrame(
    {
        "x": [ox + arrow_x / 2, ox - x_range * 0.05],
        "y": [oy - y_range * 0.05, oy + arrow_y / 2],
        "label": [xrep, yrep],
        "angle": [0, 90],
    }
)
cluster_labels = (
    adata_case.obs.groupby("annotated_cluster")[[xrep, yrep]].mean().reset_index()
)

fig = (
    gg.ggplot(adata_case.obs, gg.aes(x=xrep, y=yrep))
    + gg.geom_point(
        gg.aes(fill="annotated_cluster"), color="black", stroke=0.005, size=0.5
    )
    + gg.geom_text(
        cluster_labels,
        gg.aes(x=xrep, y=yrep, label="annotated_cluster"),
        size=5,
        color="black",
        fontweight="bold",
    )
    + gg.geom_segment(
        arrow_df,
        gg.aes(x="x", y="y", xend="xend", yend="yend"),
        arrow=gg.arrow(length=0.08, type="closed"),
        color="black",
        size=0.4,
    )
    + gg.geom_text(
        label_df,
        gg.aes(x="x", y="y", label="label", angle="angle"),
        size=8,
        color="black",
    )
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(
        axis_title=gg.element_blank(),
        axis_text=gg.element_blank(),
        axis_ticks=gg.element_blank(),
        axis_line=gg.element_blank(),
        legend_position="none",
    )
)
fig.save(
    "/workspace/experiments/06152026_retreat_content/umap_case_annotated_clusters.svg"
)

In [ ]:
fig_ = (
    fig
    + gg.theme(legend_position="right", figure_size=(6, 5))
    + gg.guides(fill=gg.guide_legend(override_aes={"size": 2}))
)
fig_.save(
    "/workspace/experiments/06152026_retreat_content/umap_case_annotated_clusters_legend.png"
)
fig_